In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install git+https://github.com/stevenpawley/Pyspatialml

In [ ]:
# Import used packages
import geopandas as gpd  # used to read the shapfile
import rasterio as rio   # used to read the raster (.tif) files
from rasterio.plot import show # used to make plots using rasterio
import matplotlib.pyplot as plt #to make plots using matplotlib
import seaborn as sns
import os
import pandas as pd

# Dataset Preparation

1. Choose the city, pollutant and year of study
2. Upload the station locations file which contain UTMY UTMX coordinates
3. Initialize the driving factors
4. Add raster files for the driving factors from external sources (eg. GEE)
5. A file called "new_points.csv" will be generated which contains the stations and remote driving factors for each month.

### 1. Choosing pollutant and year

In [ ]:
pollutant = "NO2"
year = 2019
city = "del"

### 2. Adding station locations

In [ ]:
#reading station points
points = gpd.read_file(f'/content/drive/MyDrive/DownloadTest/{city}/stations.shp')

In [ ]:
points

### 3. Initialize driving factors

In [ ]:
# Define a function to extract raster values for a given point from a TIFF file
def extract_raster_values(point, raster_file):
    longitude = point['geometry'].x
    latitude = point['geometry'].y
    with rio.open(raster_file) as src:
        row, col = src.index(longitude, latitude)
        value = src.read(1)[row, col]  # Assuming single band raster
    return value

In [ ]:
#Driving factors

points[pollutant] = None
points['Elevation'] = None
points['Rainfall'] = None
points['Population'] = None
points['VIIRS'] = None
# points['LandUse'] = None
points['Temperature'] = None
points['WindSpeed'] = None

### 4. Add driving factor raster files

In [ ]:
months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
no2_raster_files = [f'/content/drive/MyDrive/DownloadTest/{city}/no2_{year}/no2_{month}.tif' for month in months]
rainfall_raster_files = [f'/content/drive/MyDrive/DownloadTest/{city}/rainfall_{year}/rainfall_{month}.tif' for month in months]
viirs_raster_files = [f'/content/drive/MyDrive/DownloadTest/{city}/viirs_{year}/viirs_{month}.tif' for month in months]
wind_raster_files = [f'/content/drive/MyDrive/DownloadTest/{city}/windspeed_{year}/windspeed_{month}.tif' for month in months]
temp_raster_files = [f'/content/drive/MyDrive/DownloadTest/{city}/temperature_{year}/temp_{month}.tif' for month in months]

In [ ]:
NO2_raster = rio.open(f'/content/drive/MyDrive/DownloadTest/{city}/no2Test2.tif')
NO2_arr = NO2_raster.read(1)

Elevation_raster = rio.open(f'/content/drive/MyDrive/DownloadTest/{city}/elevationTest1.tif')
Elevation_arr = Elevation_raster.read(1)

Rainfall_raster = rio.open(f'/content/drive/MyDrive/DownloadTest/{city}/rainTest1.tif')
Rainfall_arr = Rainfall_raster.read(1)

Pop_raster = rio.open(f'/content/drive/MyDrive/DownloadTest/{city}/populationTest1.tif')
Pop_arr = Pop_raster.read(1)

# Landuse_raster = rio.open('/content/drive/MyDrive/DownloadTest/categoricalUse.tif')
# Landuse_arr = Landuse_raster.read(1)

In [ ]:
#Extracting the raster values to the points shapefile
count=0

for index,row in points.iterrows(): #iterate over the points in the shapefile
    longitude=row['geometry'].x #get the longitude of the point
    latitude=row['geometry'].y  #get the latitude of the point

    rowIndex, colIndex = NO2_raster.index(longitude,latitude) # the corresponding pixel to the point (longitude,latitude)

    # Extract the raster values at the point location
    points['NO2'].loc[index] = NO2_arr[rowIndex, colIndex]
    points['Elevation'].loc[index] = Elevation_arr[rowIndex, colIndex]
    points['Population'].loc[index] = Pop_arr[rowIndex, colIndex]
    points['Rainfall'].loc[index] = Rainfall_arr[rowIndex, colIndex]
    # points['LandUse'].loc[index] = Landuse_arr[rowIndex, colIndex]
    #points['VIIRS'].loc[index] = Rainfall_arr[rowIndex, colIndex]

points


### 5. Generating dataset

In [ ]:
new_rows = []

for _, point in points.iterrows():
    for month, no2_file, rainfall_file, viirs_file, temp_file, wind_file in zip(months, no2_raster_files, rainfall_raster_files, viirs_raster_files, temp_raster_files, wind_raster_files):
        no2_value = extract_raster_values(point, no2_file)
        rainfall_value = extract_raster_values(point, rainfall_file)
        viirs_value = extract_raster_values(point, viirs_file)
        temp_value = extract_raster_values(point, temp_file)
        wind_value = extract_raster_values(point, wind_file)
        new_row = {
            'geometry': point['geometry'],
            'NAME': point['NAME'],
            'NO2': no2_value,
            'Elevation': point['Elevation'],
            'Rainfall': rainfall_value,
            'Population': point['Population'],
            'VIIRS':viirs_value,
            # 'LandUse': point['LandUse'],
            'Temperature':temp_value,
            'WindSpeed':wind_value,
        }
        new_rows.append(new_row)

new_points = gpd.GeoDataFrame(new_rows, crs=points.crs)
new_points

In [ ]:
# new_points = pd.get_dummies(new_points, columns = ['LandUse'])
# new_points

In [ ]:
csv_filename = 'new_points_' + city + str(year) + '.csv'
new_points.to_csv(csv_filename, index=False)

# Adding ground data

Manually add the ground data values to corresponding rows in the *new_points* dataset generated above and upload it

In [ ]:
df=pd.read_csv(f"/content/drive/MyDrive/DownloadTest/{city}/with_ground(in)_{year}.csv", encoding = 'unicode_escape')
df

In [ ]:
# df = points

In [ ]:
df = df.rename(columns={'Monthly_avg_ground_data (µg/m³)': 'Monthly_avg_ground_data (micro g/m^3)'})

In [ ]:
df.columns.tolist()

# EDA and Preprocessing

In [ ]:
df = df.dropna()

In [ ]:
print(df.isnull().sum())

In [ ]:
df

In [ ]:
# show the correlation matric for the dataset
df_tmp = df.drop(columns = ['geometry', 'NAME','Month'], axis = 1)
corrMatrix = df_tmp.corr()
fig, ax = plt.subplots(figsize=(10,10))

sns.heatmap(corrMatrix, annot=True, linewidths=.5, ax=ax)

In [ ]:
from matplotlib import pyplot as plt

df.plot(kind='scatter', x='Monthly_avg_ground_data (micro g/m^3)', y='Elevation', s=32, alpha=.8)

plt.figure(figsize=(20, 20))

### VIIRS vs Ground Data

In [ ]:
df.plot(kind='scatter', x='Monthly_avg_ground_data (micro g/m^3)', y='VIIRS', s=32, alpha=.8)

plt.figure(figsize=(20, 20))

# Training Models

We will now run some ML models on this data to generate predictions.

#### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
import numpy as np


# Reshape the input data to be 2D arrays
y = df['Monthly_avg_ground_data (micro g/m^3)']
#x = df.drop(columns=['Monthly_avg_ground_data (micro g/m^3)',"NAME", "geometry", "Month", "LandUse_0", "LandUse_3", "LandUse_13", "LandUse_14", "LandUse_15", "LandUse_24","DSSR"])
x = df.drop(columns=['Monthly_avg_ground_data (micro g/m^3)',"NAME", "geometry", "Month"])

# Split the data into training and testing sets
#x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.7, random_state = 31)
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.7, random_state = 33)

#defining function for evaluation
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
x_train

In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn, n_bootstrap=1000, ci=95, random_state=42):
    """
    Returns mean metric and (lower, upper) CI using bootstrap resampling.
    """
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    scores = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        score = metric_fn(y_true[idx], y_pred[idx])
        scores.append(score)

    lower = np.percentile(scores, (100 - ci) / 2)
    upper = np.percentile(scores, 100 - (100 - ci) / 2)

    return np.mean(scores), lower, upper


#### Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV
#Random forest with hyperparameter tuning using GridSearch

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

np.random.seed(45)

# Define the parameter grid
param_grid = {
    'n_estimators': [15, 25, 50, 75, 100],
    'max_depth': [4, 6, 8],
    'min_samples_split': [2, 5, 10, 12],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize the Random Forest Regressor
rfr = RandomForestRegressor()

# Create the GridSearchCV object
grid_search = GridSearchCV(rfr, param_grid, cv=2)

# Train the model with hyperparameter tuning
grid_search.fit(x_train, y_train)

# Get the best model with tuned hyperparameters
best_model = grid_search.best_estimator_

# Make predictions using the best model
y_pred = best_model.predict(x_test)


# Calculate the R^2 score
r2 = r2_score(y_test, y_pred)
print("R^2 Score:", r2)

mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error (MAE):", mae)

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error (MSE):", mse)

mape = mean_absolute_percentage_error(y_test, y_pred)
print("Mean Absolute Percentage Error (MAPE):", mape)

rmse = np.sqrt(mse)
print("Root Mean Squared Error (RMSE):" , rmse)

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

# Metric wrappers (so they work with bootstrap helper)
r2_fn   = lambda y, yhat: r2_score(y, yhat)
mae_fn  = lambda y, yhat: mean_absolute_error(y, yhat)
mse_fn  = lambda y, yhat: mean_squared_error(y, yhat)
rmse_fn = lambda y, yhat: np.sqrt(mean_squared_error(y, yhat))
mape_fn = lambda y, yhat: mean_absolute_percentage_error(y, yhat)

# Bootstrap CIs
r2_mean, r2_lo, r2_hi = bootstrap_ci(y_test.values, y_pred, r2_fn)
mae_mean, mae_lo, mae_hi = bootstrap_ci(y_test.values, y_pred, mae_fn)
mse_mean, mse_lo, mse_hi = bootstrap_ci(y_test.values, y_pred, mse_fn)
rmse_mean, rmse_lo, rmse_hi = bootstrap_ci(y_test.values, y_pred, rmse_fn)
mape_mean, mape_lo, mape_hi = bootstrap_ci(y_test.values, y_pred, mape_fn)

# Pretty print
print(f"R² Score: {r2_mean:.3f} [95% CI: {r2_lo:.3f}, {r2_hi:.3f}]")
print(f"MAE: {mae_mean:.3f} [95% CI: {mae_lo:.3f}, {mae_hi:.3f}]")
print(f"MSE: {mse_mean:.3f} [95% CI: {mse_lo:.3f}, {mse_hi:.3f}]")
print(f"RMSE: {rmse_mean:.3f} [95% CI: {rmse_lo:.3f}, {rmse_hi:.3f}]")
print(f"MAPE: {mape_mean:.3f} [95% CI: {mape_lo:.3f}, {mape_hi:.3f}]")


In [ ]:
# --- STEP 1: Install SHAP (if not already installed) ---
!pip install shap

# --- STEP 2: Import SHAP and Initialize ---
import shap
import matplotlib.pyplot as plt

# Ensure JS visualization works (for some plots)
shap.initjs()

# Check which model is currently in 'best_model' and choose the right explainer
try:
    # Logic Fix: Check directly if it's a GridSearchCV object by looking for 'best_estimator_'
    if hasattr(best_model, 'best_estimator_'):
        model_to_explain = best_model.best_estimator_
    else:
        model_to_explain = best_model

    print(f"Explaining model: {type(model_to_explain).__name__}")

    # Initialize TreeExplainer
    # valid_models checks if it's a Tree-based model (RF, GBR, XGBoost)
    # If it's SVM or Linear, this line will error out and go to the 'except' block
    explainer = shap.TreeExplainer(model_to_explain)
    shap_values = explainer.shap_values(x_test)

except Exception as e:
    # Fallback to KernelExplainer for SVM or Linear Regression
    print(f"TreeExplainer failed ({e}), falling back to KernelExplainer.")

    # We use a summary of x_train to speed up calculations
    background_summary = shap.kmeans(x_train, 100)

    # For models in a Pipeline (like your SVM/Linear Regression), pass the predict function
    explainer = shap.KernelExplainer(best_model.predict, background_summary)
    shap_values = explainer.shap_values(x_test)

# --- STEP 3: Generate the Summary Plot (Reviewer's Request) ---
# This plot ranks driving factors by importance and shows their impact (positive/negative)
plt.figure(figsize=(10, 6))
plt.title("SHAP Feature Importance (Summary Plot)")
shap.summary_plot(shap_values, x_test, show=False)
plt.show()

# --- STEP 4: Generate a Bar Plot (Simpler View) ---
# This gives a clear ranking of mean importance
plt.figure(figsize=(10, 6))
plt.title("Mean |SHAP Value| (Global Feature Importance)")
shap.summary_plot(shap_values, x_test, plot_type="bar", show=False)
plt.show()

#### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Define the parameter grid for Linear Regression
param_grid = {
    'linear__fit_intercept': [True, False]
}

# Create a pipeline that includes scaling and the linear regression model
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # You can remove this if you don't want to scale features
    ('linear', LinearRegression())
])

# Create the GridSearchCV object
grid_search = GridSearchCV(pipeline, param_grid, cv=2)

# Train the model with hyperparameter tuning
grid_search.fit(x_train, y_train)

# Get the best model with tuned hyperparameters
best_model = grid_search.best_estimator_

# Make predictions using the best model
y_pred = best_model.predict(x_test)

# Calculate the R^2 score
r2 = r2_score(y_test, y_pred)
print("R^2 Score:", r2)

# Calculate additional evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error (MAE):", mae)

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error (MSE):", mse)

mape = mean_absolute_percentage_error(y_test, y_pred)
print("Mean Absolute Percentage Error (MAPE):", mape)

rmse = np.sqrt(mse)
print("Root Mean Squared Error (RMSE):", rmse)

In [ ]:
# Metric wrappers (so they work with bootstrap helper)
r2_fn   = lambda y, yhat: r2_score(y, yhat)
mae_fn  = lambda y, yhat: mean_absolute_error(y, yhat)
mse_fn  = lambda y, yhat: mean_squared_error(y, yhat)
rmse_fn = lambda y, yhat: np.sqrt(mean_squared_error(y, yhat))
mape_fn = lambda y, yhat: mean_absolute_percentage_error(y, yhat)

# Bootstrap CIs
r2_mean, r2_lo, r2_hi = bootstrap_ci(y_test.values, y_pred, r2_fn)
mae_mean, mae_lo, mae_hi = bootstrap_ci(y_test.values, y_pred, mae_fn)
mse_mean, mse_lo, mse_hi = bootstrap_ci(y_test.values, y_pred, mse_fn)
rmse_mean, rmse_lo, rmse_hi = bootstrap_ci(y_test.values, y_pred, rmse_fn)
mape_mean, mape_lo, mape_hi = bootstrap_ci(y_test.values, y_pred, mape_fn)

# Pretty print
print(f"R² Score: {r2_mean:.3f} [95% CI: {r2_lo:.3f}, {r2_hi:.3f}]")
print(f"MAE: {mae_mean:.3f} [95% CI: {mae_lo:.3f}, {mae_hi:.3f}]")
print(f"MSE: {mse_mean:.3f} [95% CI: {mse_lo:.3f}, {mse_hi:.3f}]")
print(f"RMSE: {rmse_mean:.3f} [95% CI: {rmse_lo:.3f}, {rmse_hi:.3f}]")
print(f"MAPE: {mape_mean:.3f} [95% CI: {mape_lo:.3f}, {mape_hi:.3f}]")

In [ ]:
import shap
import matplotlib.pyplot as plt

# Ensure JS visualization works
shap.initjs()

# Check which model is currently in 'best_model' and choose the right explainer
try:
    # Logic: Check if it's a GridSearchCV object (has 'best_estimator_')
    if hasattr(best_model, 'best_estimator_'):
        # If it's a GridSearch, grab the actual underlying model (e.g., Random Forest or Pipeline)
        model_to_explain = best_model.best_estimator_
    else:
        # If it's already the model (or Pipeline)
        model_to_explain = best_model

    print(f"Explaining model: {type(model_to_explain).__name__}")

    # Try TreeExplainer (Works for Random Forest, GBR)
    # This will FAIL for Pipelines (SVM/LR), sending us to the 'except' block
    explainer = shap.TreeExplainer(model_to_explain)
    shap_values = explainer.shap_values(x_test)

except Exception as e:
    print(f"TreeExplainer failed ({e}), falling back to KernelExplainer.")

    # --- FIX FOR PIPELINE ERROR ---
    # We define a wrapper function. This hides the Pipeline object from SHAP's
    # internal checks, preventing the "feature_names_in_" AttributeError.
    def prediction_wrapper(data):
        return model_to_explain.predict(data)

    # Use a small background sample for speed (100 points)
    # shap.sample is safer than kmeans for preserving DataFrame structure with Pipelines
    background_summary = shap.sample(x_train, 100)

    # Initialize KernelExplainer with the wrapper function
    explainer = shap.KernelExplainer(prediction_wrapper, background_summary)

    # Calculate SHAP values
    shap_values = explainer.shap_values(x_test)

# --- PLOTTING ---
# Generate the Summary Plot (Reviewer's Request)
plt.figure(figsize=(10, 6))
plt.title(f"SHAP Feature Importance: {type(model_to_explain).__name__}")
shap.summary_plot(shap_values, x_test, show=False)
plt.show()

# Generate the Bar Plot (Simpler View)
plt.figure(figsize=(10, 6))
plt.title(f"Mean |SHAP Value|: {type(model_to_explain).__name__}")
shap.summary_plot(shap_values, x_test, plot_type="bar", show=False)
plt.show()

#### SVM

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Define the parameter grid for SVR
param_grid = {
    'svr__C': [0.1, 1, 10, 100],
    'svr__epsilon': [0.01, 0.1, 0.2],
    'svr__kernel': ['linear', 'rbf', 'poly'],
    'svr__gamma': ['scale', 'auto']  # Only relevant for 'rbf' and 'poly' kernels
}

# Create a pipeline that includes scaling and the SVR model
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # SVR performs better with scaled features
    ('svr', SVR())
])

# Create the GridSearchCV object
grid_search = GridSearchCV(pipeline, param_grid, cv=2)

# Train the model with hyperparameter tuning
grid_search.fit(x_train, y_train)

# Get the best model with tuned hyperparameters
best_model = grid_search.best_estimator_

# Make predictions using the best model
y_pred = best_model.predict(x_test)
y_pred_svm = y_pred

# Calculate the R^2 score
r2 = r2_score(y_test, y_pred)
print("R^2 Score:", r2)

# Calculate additional evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error (MAE):", mae)

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error (MSE):", mse)

mape = mean_absolute_percentage_error(y_test, y_pred)
print("Mean Absolute Percentage Error (MAPE):", mape)

rmse = np.sqrt(mse)
print("Root Mean Squared Error (RMSE):", rmse)

In [ ]:
# Metric wrappers (so they work with bootstrap helper)
r2_fn   = lambda y, yhat: r2_score(y, yhat)
mae_fn  = lambda y, yhat: mean_absolute_error(y, yhat)
mse_fn  = lambda y, yhat: mean_squared_error(y, yhat)
rmse_fn = lambda y, yhat: np.sqrt(mean_squared_error(y, yhat))
mape_fn = lambda y, yhat: mean_absolute_percentage_error(y, yhat)

# Bootstrap CIs
r2_mean, r2_lo, r2_hi = bootstrap_ci(y_test.values, y_pred, r2_fn)
mae_mean, mae_lo, mae_hi = bootstrap_ci(y_test.values, y_pred, mae_fn)
mse_mean, mse_lo, mse_hi = bootstrap_ci(y_test.values, y_pred, mse_fn)
rmse_mean, rmse_lo, rmse_hi = bootstrap_ci(y_test.values, y_pred, rmse_fn)
mape_mean, mape_lo, mape_hi = bootstrap_ci(y_test.values, y_pred, mape_fn)

# Pretty print
print(f"R² Score: {r2_mean:.3f} [95% CI: {r2_lo:.3f}, {r2_hi:.3f}]")
print(f"MAE: {mae_mean:.3f} [95% CI: {mae_lo:.3f}, {mae_hi:.3f}]")
print(f"MSE: {mse_mean:.3f} [95% CI: {mse_lo:.3f}, {mse_hi:.3f}]")
print(f"RMSE: {rmse_mean:.3f} [95% CI: {rmse_lo:.3f}, {rmse_hi:.3f}]")
print(f"MAPE: {mape_mean:.3f} [95% CI: {mape_lo:.3f}, {mape_hi:.3f}]")

In [ ]:
import shap
import matplotlib.pyplot as plt

# Ensure JS visualization works
shap.initjs()

# Check which model is currently in 'best_model' and choose the right explainer
try:
    # Logic: Check if it's a GridSearchCV object (has 'best_estimator_')
    if hasattr(best_model, 'best_estimator_'):
        # If it's a GridSearch, grab the actual underlying model (e.g., Random Forest or Pipeline)
        model_to_explain = best_model.best_estimator_
    else:
        # If it's already the model (or Pipeline)
        model_to_explain = best_model

    print(f"Explaining model: {type(model_to_explain).__name__}")

    # Try TreeExplainer (Works for Random Forest, GBR)
    # This will FAIL for Pipelines (SVM/LR), sending us to the 'except' block
    explainer = shap.TreeExplainer(model_to_explain)
    shap_values = explainer.shap_values(x_test)

except Exception as e:
    print(f"TreeExplainer failed ({e}), falling back to KernelExplainer.")

    # --- FIX FOR PIPELINE ERROR ---
    # We define a wrapper function. This hides the Pipeline object from SHAP's
    # internal checks, preventing the "feature_names_in_" AttributeError.
    def prediction_wrapper(data):
        return model_to_explain.predict(data)

    # Use a small background sample for speed (100 points)
    # shap.sample is safer than kmeans for preserving DataFrame structure with Pipelines
    background_summary = shap.sample(x_train, 100)

    # Initialize KernelExplainer with the wrapper function
    explainer = shap.KernelExplainer(prediction_wrapper, background_summary)

    # Calculate SHAP values
    shap_values = explainer.shap_values(x_test)

# --- PLOTTING ---
# Generate the Summary Plot (Reviewer's Request)
plt.figure(figsize=(10, 6))
plt.title(f"SHAP Feature Importance: {type(model_to_explain).__name__}")
shap.summary_plot(shap_values, x_test, show=False)
plt.show()

# Generate the Bar Plot (Simpler View)
plt.figure(figsize=(10, 6))
plt.title(f"Mean |SHAP Value|: {type(model_to_explain).__name__}")
shap.summary_plot(shap_values, x_test, plot_type="bar", show=False)
plt.show()

#### Gradient Boosting Regressor

In [ ]:
#Gradient Boosting regressor with hyperparameter tuning using GridSearch

# Define the parameter grid
param_grid = {
    'n_estimators': [25, 50, 100, 200],
    'learning_rate': [0.1, 0.01, 0.001],
    'max_depth': [3, 5, 8, 12, 16]
}

# Initialize the Gradient Boosting Regressor
gbr = GradientBoostingRegressor()

# Create the GridSearchCV object
grid_search = GridSearchCV(gbr, param_grid, cv=2)

# Train the model with hyperparameter tuning
grid_search.fit(x_train, y_train)

# Get the best model with tuned hyperparameters
best_model = grid_search.best_estimator_

# Make predictions using the best model
y_pred = best_model.predict(x_test)
y_pred_gbr = y_pred


# Calculate the R^2 score
r2 = r2_score(y_test, y_pred)
print("R^2 Score:", r2)

mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error (MAE):", mae)

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error (MSE):", mse)

mape = mean_absolute_percentage_error(y_test, y_pred)
print("Mean Absolute Percentage Error (MAPE):", mape)

rmse = np.sqrt(mse)
print("Root Mean Squared Error (RMSE):" , rmse)

In [ ]:
# Metric wrappers (so they work with bootstrap helper)
r2_fn   = lambda y, yhat: r2_score(y, yhat)
mae_fn  = lambda y, yhat: mean_absolute_error(y, yhat)
mse_fn  = lambda y, yhat: mean_squared_error(y, yhat)
rmse_fn = lambda y, yhat: np.sqrt(mean_squared_error(y, yhat))
mape_fn = lambda y, yhat: mean_absolute_percentage_error(y, yhat)

# Bootstrap CIs
r2_mean, r2_lo, r2_hi = bootstrap_ci(y_test.values, y_pred, r2_fn)
mae_mean, mae_lo, mae_hi = bootstrap_ci(y_test.values, y_pred, mae_fn)
mse_mean, mse_lo, mse_hi = bootstrap_ci(y_test.values, y_pred, mse_fn)
rmse_mean, rmse_lo, rmse_hi = bootstrap_ci(y_test.values, y_pred, rmse_fn)
mape_mean, mape_lo, mape_hi = bootstrap_ci(y_test.values, y_pred, mape_fn)

# Pretty print
print(f"R² Score: {r2_mean:.3f} [95% CI: {r2_lo:.3f}, {r2_hi:.3f}]")
print(f"MAE: {mae_mean:.3f} [95% CI: {mae_lo:.3f}, {mae_hi:.3f}]")
print(f"MSE: {mse_mean:.3f} [95% CI: {mse_lo:.3f}, {mse_hi:.3f}]")
print(f"RMSE: {rmse_mean:.3f} [95% CI: {rmse_lo:.3f}, {rmse_hi:.3f}]")
print(f"MAPE: {mape_mean:.3f} [95% CI: {mape_lo:.3f}, {mape_hi:.3f}]")

In [ ]:
from scipy.stats import ttest_rel

svm_errors = np.abs(y_test.values - y_pred_svm)
gbr_errors = np.abs(y_test.values - y_pred_gbr)

t_stat, p_value = ttest_rel(svm_errors, gbr_errors)

print(f"Paired t-test (MAE errors): t = {t_stat:.3f}, p = {p_value:.4f}")


In [ ]:
# --- STEP 2: Import SHAP and Initialize ---
import shap
import matplotlib.pyplot as plt

# Ensure JS visualization works (for some plots)
shap.initjs()

# Check which model is currently in 'best_model' and choose the right explainer
# Note: TreeExplainer is faster for Random Forest/GBR. KernelExplainer is for SVM/others.
try:
    # Try using TreeExplainer (Works for Random Forest, Gradient Boosting, XGBoost)
    # We access the actual model inside the GridSearch wrapper if necessary
    if hasattr(best_model, 'estimator'): # For GridSearchCV objects not strictly unwrapped
        model_to_explain = best_model.best_estimator_
    else:
        model_to_explain = best_model

    print(f"Explaining model: {type(model_to_explain).__name__}")

    explainer = shap.TreeExplainer(model_to_explain)
    shap_values = explainer.shap_values(x_test)

except Exception as e:
    # Fallback to KernelExplainer for SVM or Linear Regression
    # Note: KernelExplainer is slower, so we use a summary (subset) of the background data
    print(f"TreeExplainer failed ({e}), falling back to KernelExplainer (slower).")

    # We use a summary of x_train to speed up calculations (e.g., 100 samples)
    background_summary = shap.kmeans(x_train, 100)

    # For models in a Pipeline (like your SVM/Linear Regression), pass the predict function
    explainer = shap.KernelExplainer(best_model.predict, background_summary)
    shap_values = explainer.shap_values(x_test)

# --- STEP 3: Generate the Summary Plot (Reviewer's Request) ---
# This plot ranks driving factors by importance and shows their impact (positive/negative)
plt.figure(figsize=(10, 6))
plt.title("SHAP Feature Importance (Summary Plot)")
shap.summary_plot(shap_values, x_test, show=False)
plt.show()

# --- STEP 4: Generate a Bar Plot (Simpler View) ---
# This gives a clear ranking of mean importance
plt.figure(figsize=(10, 6))
plt.title("Mean |SHAP Value| (Global Feature Importance)")
shap.summary_plot(shap_values, x_test, plot_type="bar", show=False)
plt.show()

# Visualizing predictions for the entire city

### Grid Creation

In [ ]:
import numpy as np
import pandas as pd
from pyproj import Proj, transform

# Define the boundaries of the city in lat and lon
lat_min, lat_max = 12.85, 13.20  # Example for a city like Bangalore
lon_min, lon_max = 77.45, 77.80

# lat_min, lat_max = 28.40, 28.88  # Latitude range for Delhi
# lon_min, lon_max = 76.84, 77.34  # Longitude range for Delhi

# Define the grid resolution (distance between points in degrees)
grid_size = 0.01  # Adjust as needed

# Create the grid points
lat_grid = np.arange(lat_min, lat_max, grid_size)
lon_grid = np.arange(lon_min, lon_max, grid_size)
grid_points = [(lat, lon) for lat in lat_grid for lon in lon_grid]

# Define the projection: WGS84 (lat/lon) to UTM (Zone 43N, for example)
proj_wgs84 = Proj(proj='latlong', datum='WGS84')
proj_utm = Proj(proj='utm', zone=43, datum='WGS84')

# Convert grid points to UTM coordinates and keep lat/lon
utm_grid_points = [(lat, lon, *transform(proj_wgs84, proj_utm, lon, lat)) for lat, lon in grid_points]

# Convert to DataFrame and round the UTM coordinates to the nearest integer
grid_df = pd.DataFrame(utm_grid_points, columns=['latitude', 'longitude', 'UTMX', 'UTMY'])
grid_df[['UTMX', 'UTMY']] = grid_df[['UTMX', 'UTMY']].round(0).astype(int)

grid_df

In [ ]:
grid_df['NO2'] = None
grid_df['Elevation'] = None
grid_df['Rainfall'] = None
grid_df['Population'] = None
grid_df['VIIRS'] = None
# grid_df['LandUse'] = None
grid_df['Temperature'] = None
grid_df['WindSpeed'] = None
grid_df

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
# no2_raster_files = [f'/content/drive/MyDrive/DownloadTest/no2_{month}.tif' for month in months]
# rainfall_raster_files = [f'/content/drive/MyDrive/DownloadTest/rainfall_{month}.tif' for month in months]
# viirs_raster_files = [f'/content/drive/MyDrive/DownloadTest/viirs_{month}.tif' for month in months]
# wind_raster_files = [f'/content/drive/MyDrive/DownloadTest/windspeed_{month}.tif' for month in months]
# temp_raster_files = [f'/content/drive/MyDrive/DownloadTest/temp_{month}.tif' for month in months]

NO2_raster = rio.open('/content/drive/MyDrive/DownloadTest/blr/no2_2022/NO2_blr.tif')
NO2_arr = NO2_raster.read(1)

Elevation_raster = rio.open('/content/drive/MyDrive/DownloadTest/elevation_blr_sq.tif')
Elevation_arr = Elevation_raster.read(1)

Rainfall_raster = rio.open('/content/drive/MyDrive/DownloadTest/blr/rainfall_2022/rainfall_blr.tif')
Rainfall_arr = Rainfall_raster.read(1)

Pop_raster = rio.open('/content/drive/MyDrive/DownloadTest/population_blr_sq.tif')
Pop_arr = Pop_raster.read(1)

Viirs_raster = rio.open('/content/drive/MyDrive/DownloadTest/blr/viirs_2022/viirs_blr.tif')
Viirs_arr = Viirs_raster.read(1)

Temperature_raster = rio.open('/content/drive/MyDrive/DownloadTest/blr/temperature_2022/temperature_blr.tif')
Temperature_arr = Temperature_raster.read(1)

Windspeed_raster = rio.open('/content/drive/MyDrive/DownloadTest/blr/windspeed_2022/wind_blr.tif')
Windspeed_arr = Windspeed_raster.read(1)

# Landuse_raster = rio.open('/content/drive/MyDrive/DownloadTest/categoricalUse.tif')
# Landuse_arr = Landuse_raster.read(1)

#Extracting the raster values to the points shapefile
count=0

for index,row in grid_df.iterrows(): #iterate over the points in the shapefile
    longitude=row['UTMX'] #get the longitude of the point
    latitude=row['UTMY']  #get the latitude of the point

    rowIndex, colIndex = NO2_raster.index(longitude,latitude) # the corresponding pixel to the point (longitude,latitude)

    # Extract the raster values at the point location
    grid_df['NO2'].loc[index] = NO2_arr[rowIndex, colIndex]
    grid_df['Elevation'].loc[index] = Elevation_arr[rowIndex, colIndex]
    grid_df['Population'].loc[index] = Pop_arr[rowIndex, colIndex]
    grid_df['Rainfall'].loc[index] = Rainfall_arr[rowIndex, colIndex]
    grid_df['VIIRS'].loc[index] = Viirs_arr[rowIndex, colIndex]
    grid_df['Temperature'].loc[index] = Temperature_arr[rowIndex, colIndex]
    grid_df['WindSpeed'].loc[index] = Windspeed_arr[rowIndex, colIndex]
    # points['LandUse'].loc[index] = Landuse_arr[rowIndex, colIndex]
    #points['VIIRS'].loc[index] = Rainfall_arr[rowIndex, colIndex]


In [ ]:
grid_df

In [ ]:
null_percentage = grid_df.isnull().mean() * 100
print(null_percentage)

In [ ]:
# Assuming df is your DataFrame
grid_df = grid_df.dropna()
grid_df

In [ ]:
# pip install earthengine-api

In [ ]:
# import ee
# ee.Authenticate()  # Run this only once to authenticate

In [ ]:
# ee.Initialize(project='ee-brijdesai2003')

In [ ]:
# no2_dataset = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2') \
#               .filterDate('2023-01-01', '2023-01-31') \
#               .select('tropospheric_NO2_column_number_density') \
#               .mean()  # Get the average for the month

# elevation_dataset = ee.Image('CGIAR/SRTM90_V4')

# rainfall_dataset = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE") \
#                   .filterDate('2023-01-01', '2023-01-31') \
#                   .select('pr') \
#                   .mean()

# population_dataset = ee.ImageCollection("CIESIN/GPWv411/GPW_Population_Count") \
#                         .filterDate('2020-01-01', '2020-12-31') \
#                         .mean()

# viirs_dataset = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG') \
#                         .filterDate('2023-01-01', '2023-01-31') \
#                         .select('avg_rad') \
#                         .mean()

# # landuse_dataset = ee.Image("JRC/GHSL/P2023A/GHS_BUILT_C/2018")

# temperature_dataset = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE") \
#                   .filterDate('2023-01-01', '2023-01-31') \
#                   .select('tmmx') \
#                   .mean()

# windspeed_dataset = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE") \
#                   .filterDate('2023-01-01', '2023-01-31') \
#                   .select('vs') \
#                   .mean()

In [ ]:
# # Define a function to get data from Earth Engine for a single point
# def get_ee_data(lat, lon):
#     point = ee.Geometry.Point(lon, lat)

#     # Get NO2 data from TROPOMI
#     no2_value = no2_dataset.reduceRegion(
#         reducer=ee.Reducer.mean(),
#         geometry=point,
#         scale=3000
#     ).get('tropospheric_NO2_column_number_density').getInfo()

#     # Example: Get Elevation data
#     elevation_value = elevation_dataset.reduceRegion(
#         reducer=ee.Reducer.mean(),
#         geometry=point,
#         scale=3000
#     ).get('elevation').getInfo()

#     # Get Rainfall data from TerraClimate
#     rainfall_value = rainfall_dataset.reduceRegion(
#         reducer=ee.Reducer.mean(),
#         geometry=point,
#         scale=3000
#     ).get('pr').getInfo()

#     # Get Population data from CIESIN

#     population_value = population_dataset.reduceRegion(
#         reducer=ee.Reducer.mean(),
#         geometry=point,
#         scale=3000
#     ).get('population_count').getInfo()

#     # Get VIIRS Nighttime Lights data
#     viirs_value = viirs_dataset.reduceRegion(
#         reducer=ee.Reducer.mean(),
#         geometry=point,
#         scale=3000
#     ).get('avg_rad').getInfo()

#      # Get Land Use data from GHS Built-Up Grid
#     # landuse_value = landuse_dataset.reduceRegion(
#     #     reducer=ee.Reducer.mean(),
#     #     geometry=point,
#     #     scale=1000
#     # ).get('built_characteristics').getInfo()

#     temperature_value = temperature_dataset.reduceRegion(
#         reducer=ee.Reducer.mean(),
#         geometry=point,
#         scale=3000
#     ).get('tmmx').getInfo()

#     windspeed_value = windspeed_dataset.reduceRegion(
#         reducer=ee.Reducer.mean(),
#         geometry=point,
#         scale=3000
#     ).get('vs').getInfo()

#     return no2_value, elevation_value, rainfall_value, population_value, viirs_value, temperature_value, windspeed_value

#     # Add other data sources similarly...

In [ ]:
# temp = grid_df.head(3)
# temp

In [ ]:
# # Apply the function to all grid points
# temp[['Tropomi_NO2', 'Elevation', 'Rainfall', 'Population', 'viirs', 'Temperature', 'Windspeed']] = temp.apply(lambda row: get_ee_data(row['latitude'], row['longitude']), axis=1, result_type='expand')
# temp

In [ ]:
# grid_df[['NO2 (mol/m^2)', 'Elevation', 'Rainfall', 'Population', 'VIIRS', 'Temperature', 'WindSpeed']] = grid_df.apply(lambda row: get_ee_data(row['latitude'], row['longitude']), axis=1, result_type='expand')

In [ ]:
csv_filename = 'blr_2022.csv'
grid_df.to_csv(csv_filename, index=False)

In [ ]:
# Assuming df is your DataFrame and you want to rename 'old_column' to 'new_column'
grid_df.rename(columns={'NO2': 'NO2 (mol/m^2)'}, inplace=True)

In [ ]:
grid_df.head()

In [ ]:
features = ['NO2 (mol/m^2)', 'Elevation', 'Rainfall', 'Population', 'VIIRS', 'Temperature', 'WindSpeed']
grid_df['NO2_prediction'] = best_model.predict(grid_df[features])

In [ ]:
grid_df.head()

In [ ]:
import folium
from folium.plugins import HeatMap

# Create a base map centered around the city
m = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2], zoom_start=12)

# Convert predictions to a list of [latitude, longitude, NO2] for HeatMap
heat_data = [[row['latitude'], row['longitude'], row['NO2_prediction']] for index, row in grid_df.iterrows()]

# Add the heatmap layer
HeatMap(heat_data, radius=15).add_to(m)

# Save map to HTML file or display directly
m.save('no2_heatmap.html')
m

In [ ]:
import folium
from folium.plugins import HeatMap

# Create a base map centered around the city
m = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2], zoom_start=12)

# Convert predictions to a list of [latitude, longitude, NO2] for HeatMap
heat_data = [[row['latitude'], row['longitude'], row['NO2_prediction']] for index, row in grid_df.iterrows()]

# Add the heatmap layer with NO2 predictions
HeatMap(heat_data, radius=20, blur=25, max_zoom=12, min_opacity=0.4).add_to(m)

# HTML for the custom legend
legend_html = '''
<div style="
    position: fixed;
    bottom: 50px; left: 50px; width: 150px; height: 150px;
    background-color: white; border:2px solid grey; z-index:9999; font-size:14px;
    padding: 0px;
    ">
    <b>NO2 Levels</b><br>
    <i style="background: rgba(0, 0, 255, 0.5);width: 20px;height: 10px;display: inline-block;"></i> Low (<10 μg/m³)<br>
    <i style="background: rgba(0, 255, 0, 0.5);width: 20px;height: 10px;display: inline-block;"></i> Moderate (10-20 μg/m³)<br>
    <i style="background: rgba(255, 255, 0, 0.5);width: 20px;height: 10px;display: inline-block;"></i> High (20-40 μg/m³)<br>
    <i style="background: rgba(255, 0, 0, 0.5);width: 20px;height: 10px;display: inline-block;"></i> Very High (>40 μg/m³)
</div>
'''

# Add the custom legend to the map
m.get_root().html.add_child(folium.Element(legend_html))

# Save the map to an HTML file or display directly
m.save('no2_heatmap_with_legend.html')
m


In [ ]:
import plotly.express as px

# Prepare data for Plotly
df = grid_df[['latitude', 'longitude', 'NO2_prediction']]

# Create a scatter mapbox plot with a color scale for NO2 predictions
fig = px.scatter_mapbox(df, lat='latitude', lon='longitude',
                        color='NO2_prediction',
                        color_continuous_scale='Viridis',
                        mapbox_style='open-street-map',
                        zoom=12)

# Customize the map
fig.update_layout(mapbox_center={"lat": (lat_min + lat_max) / 2, "lon": (lon_min + lon_max) / 2})

# Display the map
fig.show()


### Kernel Density Estimation

In [ ]:
from sklearn.neighbors import KernelDensity
from scipy.stats import gaussian_kde

# Assuming your DataFrame is already loaded as grid_df

# Extract the latitude, longitude, and NO2_prediction columns
coords = grid_df[['latitude', 'longitude']].values
no2_values = grid_df['NO2_prediction'].values

# Define the bandwidth for the KDE
bandwidth = 0.01  # You can experiment with this value for smoothing

# Fit a KDE model using the latitude and longitude data
kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian')
kde.fit(coords, sample_weight=no2_values)

# Create a grid of points over the map (adjust the step size for smoother results)
lat_min, lat_max = grid_df['latitude'].min(), grid_df['latitude'].max()
lon_min, lon_max = grid_df['longitude'].min(), grid_df['longitude'].max()

# Define the grid size
lat_lin = np.linspace(lat_min, lat_max, 100)
lon_lin = np.linspace(lon_min, lon_max, 100)
lon_grid, lat_grid = np.meshgrid(lon_lin, lat_lin)
grid_points = np.vstack([lat_grid.ravel(), lon_grid.ravel()]).T

# Compute the KDE for each point on the grid
log_density = kde.score_samples(grid_points)
density = np.exp(log_density).reshape(lat_grid.shape)

# Normalize the density for better visualization
density = (density - density.min()) / (density.max() - density.min())

# Prepare the heatmap data by associating each grid point with its density
heat_data = []
for i in range(lat_grid.shape[0]):
    for j in range(lat_grid.shape[1]):
        heat_data.append([lat_grid[i, j], lon_grid[i, j], density[i, j]])

# HTML for the custom legend
legend_html = '''
<div style="
    position: fixed;
    bottom: 50px; left: 50px; width: 150px; height: 150px;
    background-color: white; border:2px solid grey; z-index:9999; font-size:14px;
    padding: 0px;
    ">
    <b>NO2 Levels</b><br>
    <i style="background: rgba(0, 0, 255, 0.5);width: 20px;height: 10px;display: inline-block;"></i> Low (<10 μg/m³)<br>
    <i style="background: rgba(0, 255, 0, 0.5);width: 20px;height: 10px;display: inline-block;"></i> Moderate (10-20 μg/m³)<br>
    <i style="background: rgba(255, 255, 0, 0.5);width: 20px;height: 10px;display: inline-block;"></i> High (20-40 μg/m³)<br>
    <i style="background: rgba(255, 0, 0, 0.5);width: 20px;height: 10px;display: inline-block;"></i> Very High (>40 μg/m³)
</div>
'''

# Create a folium map
map_center = [grid_df['latitude'].mean(), grid_df['longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=12)

# Add the KDE heatmap to the map
HeatMap(heat_data, radius=18, blur=20, max_zoom=1).add_to(m)

# Add the custom legend to the map
m.get_root().html.add_child(folium.Element(legend_html))

# Save the map to an HTML file or display it
m.save("NO2_prediction_kde_heatmap.html")

# If you're in a notebook environment, you can display the map directly
m

In [ ]:
# import datashader as ds
# import datashader.transfer_functions as tf
# import pandas as pd
# from bokeh.plotting import show
# from bokeh.tile_providers import get_provider, CARTODBPOSITRON

# # Create a Canvas
# cvs = ds.Canvas(plot_width=800, plot_height=600, x_range=(lon_min, lon_max), y_range=(lat_min, lat_max))

# # Create the heatmap using datashader
# agg = cvs.points(grid_df, 'longitude', 'latitude', agg=ds.mean('NO2_prediction'))

# # Convert aggregation to image
# img = tf.shade(agg, cmap=["green", "yellow", "red"])

# # Display the map
# show(img)


In [ ]:
# from keplergl import KeplerGl

# # Initialize the map
# map_1 = KeplerGl()

# # Add your data
# map_1.add_data(data=grid_df, name="NO2 Data")

# # Show the map in the notebook
# map_1
